In [ ]:
using Pkg
Pkg.activate("/Users/bursche/Documents/GitHub/JPEC_BCRIT")
Base.active_project()

In [ ]:
using GeneralizedPerturbedEquilibrium
using GeneralizedPerturbedEquilibrium: GGJParameters
using GeneralizedPerturbedEquilibrium
using GeneralizedPerturbedEquilibrium: InnerLayer
using GeneralizedPerturbedEquilibrium.InnerLayer: solve_inner
import GeneralizedPerturbedEquilibrium.InnerLayer.GGJ: sfac


In [ ]:
using Plots
default(
    fontfamily="Georgia",
    margin=12Plots.mm,
    size=(800, 500),
    dpi=150
)

In [ ]:
struct TorqueBalance
    model
    params
    Q0::Float64
    P::Float64
    delta_n_p::Float64
    lu::Float64
    sval::Float64
end

function torque_balance_value(tb::TorqueBalance, Q::Number)
    Δ = solve_inner(tb.model, tb.params, ComplexF64(Q)).tearing
    jxb = -imag(1.0 / (Δ + tb.delta_n_p))
    return 2.0 * tb.P * (tb.Q0 - Q) / jxb
end

In [ ]:
# You can substitute any valid SLAYERParameters you already have.
# This is just an example skeleton.
p = GeneralizedPerturbedEquilibrium.InnerLayer.slayer_parameters(
    n_e=1e19, t_e=1e3, t_i=1e3,
    omega=0.0, omega_e=4, omega_i=-2,
    qval=2.0, sval_r=0.5, bt=2.0, rs=1.0, R0=3.0, mu_i=2.0, zeff=1.0,
    chi_perp=1.0, chi_tor=1.0, m=2, n=1
)

tb = TorqueBalance(
    GeneralizedPerturbedEquilibrium.InnerLayer.SLAYERModel(;),
    p,
    0.5,
    1.0,
    1e-2,
    p.lu,
    p.sval_r
)

In [ ]:
p_ggj = GeneralizedPerturbedEquilibrium.InnerLayer.glasser_wang_2020_eq55()
lu_ggj = sfac(p_ggj)
sval_ggj = 0.5

tb = TorqueBalance(
    GeneralizedPerturbedEquilibrium.InnerLayer.GGJModel(; solver=:ray),
    p_ggj,
    0.5,
    1.0,
    1e-2,
    lu_ggj,
    sval_ggj
)

In [ ]:
"""

tau_h=R0*(mu0*rho)**0.5/(nns*sval*bt) ! alfven time across surface
lu=tau_r/tau_h                   ! Lundquist number
Qconv=lu**(1.0/3.0)*tau_h        ! conversion to Qs based on Cole

! note Q depends on Qconv even if omega is fixed.
Q=Qconv*omega
Q_e=-Qconv*omega_e
Q_i=-Qconv*omega_i

"""
2=-lu**(1.0/3.0)*tau_h *OE
OE = -2 

In [ ]:
Qs = range(-5.0, 5.0, length=200)

vals = [torque_balance_value(tb, q) for q in Qs]

In [ ]:
plot(Qs, real.(vals), label="Re(balance)", lw=2)
plot!(Qs, imag.(vals), label="Im(balance)", lw=2)
xlabel!("Q")
ylabel!("torque balance")
title!("tb(Q)")

In [ ]:
Δs = [solve_inner(tb.model, tb.params, ComplexF64(q)).tearing for q in Qs]
jxbs = [-imag(1.0 / (d + tb.delta_n_p)) for d in Δs]

p1 = plot(Qs, real.(Δs), label="Re(Δ)", lw=2)
plot!(p1, Qs, imag.(Δs), label="Im(Δ)", lw=2)
xlabel!(p1, "Q")
ylabel!(p1, "Δ")
title!(p1, "Inner-layer Δ(Q)")

p2 = plot(Qs, jxbs, label="jxb", lw=2)
xlabel!(p2, "Q")
ylabel!(p2, "jxb")
title!(p2, "jxb(Q) = -Im[1/(Δ + δ_n_p)]")

p3 = plot(Qs, real.(vals), label="Re(balance)", lw=2)
plot!(p3, Qs, imag.(vals), label="Im(balance)", lw=2)
xlabel!(p3, "Q")
ylabel!(p3, "balance")
title!(p3, "2P(Q0-Q)/jxb")

plot(p1, p2, p3, layout=(3,1), size=(800, 1000))